# Vaani-FastConformer-Hindi — Kathbath (Vistaar) Test-Set Evaluation

Evaluates `ARTPARK-IISc/Vaani-FastConformer-Hindi` (NeMo FastConformer, CTC) on the
**Kathbath** benchmark test set (via the official Vistaar release) for **Hindi**.

**No manual dataset upload needed** - the notebook downloads the official combined
benchmark zip directly from AI4Bharat's object store and extracts only the language you
need. The Hindi folder has a `manifest.json` (NeMo-style JSONL: one
`{"audio_filepath", "duration", "text"}` object per line) and a `wavs/` folder of audio.

**Same method as the IndicConformer / IndicWav2Vec Kathbath notebooks:** auto-download +
selective extraction of the Vistaar zip, a manifest reader that's robust to whatever
subfolder naming Kaggle mounts the data under, per-sample checkpointing with resume, and
WER/CER via `jiwer`.

**Model-specific note:** Vaani-FastConformer is loaded through NeMo's
`ASRModel.from_pretrained`. Some checkpoints in this family ship weights in `bf16`, which
Tesla T4/P100 GPUs don't handle reliably for all ops - so the model is loaded on CPU first,
cast to `float32`, and only then moved to GPU.

**Before running:**
1. (Optional) Set an HF token in the login cell - only needed if you're hitting HF rate
   limits pulling the model.
2. Run cells top to bottom - the download/extract cell handles Hindi automatically.


In [22]:
# This Python 3 environment comes with many helpful analytics libraries installed
import numpy as np
import pandas as pd
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames[:5]:
        print(os.path.join(dirname, filename))


In [23]:
!nvidia-smi


Wed Jul 22 07:00:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   77C    P0             32W /   70W |    2575MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [24]:
%%capture
# NeMo + ASR extras (~4-5 min first run; cached on re-run)
!pip install Cython
!pip install 'nemo_toolkit[asr]'
!pip install -q jiwer
print('Dependencies installed')


## Download the Kathbath (Vistaar) benchmark automatically

No manual upload needed. AI4Bharat hosts the Vistaar benchmark test sets as a single combined
zip covering all 12 languages: `kathbath.zip` from
https://github.com/AI4Bharat/vistaar (see "Download Training Datasets and Benchmarks").

The zip's internal layout is `kathbath/<lang>/manifest.json` + `kathbath/<lang>/wavs/*.wav`,
so no manifest editing is needed here either.

Since it's one zip for all languages, the cell below downloads the whole archive once
(cached - re-running skips re-download), then **selectively extracts only the languages
listed in `LANGS_TO_EXTRACT`** so you're not paying disk/time for languages you're not
using yet. Starting with Hindi only.


In [25]:
import os

ZIP_URL = "https://indicwhisper.objectstore.e2enetworks.net/vistaar_benchmarks/kathbath.zip"
DOWNLOAD_DIR = "/kaggle/working/kathbath_download"
ZIP_PATH = os.path.join(DOWNLOAD_DIR, "kathbath.zip")

os.makedirs(DOWNLOAD_DIR, exist_ok=True)


def zip_is_valid(path):
    """Returns True only if the file exists AND is a complete, openable zip archive.
    A partial/interrupted download can still have a valid-looking header, so we
    actually try to open it and read the central directory."""
    if not os.path.exists(path) or os.path.getsize(path) == 0:
        return False
    try:
        import zipfile
        with zipfile.ZipFile(path, "r") as zf:
            bad_file = zf.testzip()  # returns None if all entries check out
            return bad_file is None
    except Exception:
        return False


if zip_is_valid(ZIP_PATH):
    print(f"Zip already downloaded and verified at {ZIP_PATH}, skipping download.")
else:
    if os.path.exists(ZIP_PATH):
        print(f"Found an existing file at {ZIP_PATH} but it's incomplete/corrupt "
              f"(size={os.path.getsize(ZIP_PATH) / 1e9:.2f} GB) - resuming download.")
    else:
        print("Downloading Kathbath (Vistaar benchmark) zip - single archive covering all 12")
        print("languages, so this can take a while depending on connection speed...")
    # -c resumes from where a partial file left off instead of restarting from zero
    !wget -c -O {ZIP_PATH} {ZIP_URL}

    if not zip_is_valid(ZIP_PATH):
        raise RuntimeError(
            f"Download finished but {ZIP_PATH} is still not a valid/complete zip "
            f"(size={os.path.getsize(ZIP_PATH) / 1e9:.2f} GB). This usually means the "
            f"session was interrupted again, or the server doesn't support resumable "
            f"range requests for this URL. Try re-running this cell (wget -c will keep "
            f"resuming), or delete {ZIP_PATH} to force a full fresh download."
        )
    print("Zip verified OK.")

print(f"Zip size: {os.path.getsize(ZIP_PATH) / 1e9:.2f} GB")


Zip already downloaded and verified at /kaggle/working/kathbath_download/kathbath.zip, skipping download.
Zip size: 3.46 GB


In [26]:
import zipfile
from tqdm import tqdm

EXTRACT_ROOT = "/kaggle/working/kathbath_data"
os.makedirs(EXTRACT_ROOT, exist_ok=True)

# Languages to extract right now. Add more later ("bengali", "tamil", "marathi", "telugu")
# and re-run this cell to bring in additional languages without re-downloading.
LANGS_TO_EXTRACT = ["hindi"]

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    all_names = zf.namelist()
    top_level = all_names[0].split("/")[0]
    print(f"Top-level folder inside zip: '{top_level}'")

    already_extracted = {
        lang for lang in LANGS_TO_EXTRACT
        if os.path.exists(os.path.join(EXTRACT_ROOT, top_level, lang, "manifest.json"))
    }
    to_process = [lang for lang in LANGS_TO_EXTRACT if lang not in already_extracted]
    if already_extracted:
        print(f"Already extracted, skipping: {sorted(already_extracted)}")
    if not to_process:
        print("Nothing new to extract.")
    else:
        to_extract = [
            n for n in all_names
            if any(n.startswith(f"{top_level}/{lang}/") for lang in to_process)
        ]
        print(f"Extracting {len(to_extract)} files for: {to_process} ...")
        for name in tqdm(to_extract):
            zf.extract(name, EXTRACT_ROOT)

KATHBATH_ROOT = os.path.join(EXTRACT_ROOT, top_level)
print(f"\nKATHBATH_ROOT = {KATHBATH_ROOT}")
print(f"Contents: {os.listdir(KATHBATH_ROOT)}")


Top-level folder inside zip: 'kathbath'
Already extracted, skipping: ['hindi']
Nothing new to extract.

KATHBATH_ROOT = /kaggle/working/kathbath_data/kathbath
Contents: ['hindi']


In [27]:
from huggingface_hub import login
import os

# SECURITY: don't hardcode HF tokens in the notebook. Use Kaggle's built-in
# "Add-ons > Secrets" (or an environment variable) so the token isn't stored in
# plain text in the .ipynb file, which is easy to accidentally share/commit.
hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    try:
        from kaggle_secrets import UserSecretsClient
        hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        hf_token = None

if hf_token:
    login(token=hf_token)
    print("Successfully logged into Hugging Face Hub!")
else:
    print("No HF token found - continuing without login.")


Successfully logged into Hugging Face Hub!


In [28]:
import torch
import nemo.collections.asr as nemo_asr

MODEL_ID = "ARTPARK-IISc/Vaani-FastConformer-Hindi"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {device}")
if device == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    # T4/P100 don't handle bf16 reliably for all ops; force float32
    torch.set_default_dtype(torch.float32)
    torch.backends.cuda.matmul.allow_tf32 = False

# Load on CPU first to avoid CUDA dtype assertion during weight loading
print("Loading model on CPU ...")
model = nemo_asr.models.ASRModel.from_pretrained(
    model_name=MODEL_ID,
    map_location="cpu",
)

# Cast all parameters to float32 (checkpoint may contain bf16 tensors)
model = model.float()

# Move to GPU
model = model.to(device)
model.eval()
print(f"Model loaded on {device} as float32")


[NeMo W 2026-07-22 07:01:49 megatron_init:62] Megatron num_microbatches_calculator not found, using Apex version.
OneLogger: Setting error_handling_strategy to DISABLE_QUIETLY_AND_REPORT_METRIC_ERROR for rank (rank=0) with OneLogger disabled. To override: explicitly set error_handling_strategy parameter.
No exporters were provided. This means that no telemetry data will be collected.
[NeMo W 2026-07-22 07:01:52 nemo_logging:364] /usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
      m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
    
[NeMo W 2026-07-22 07:01:52 nemo_logging:364] /usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
      m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
    
[NeMo W 2026-07-22 07:01:52 nemo_logging:364] /usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
      elif re.

Device : cuda
GPU    : Tesla T4
Loading model on CPU ...


Vaani-FastConformer-Hindi.nemo:   0%|          | 0.00/1.75G [00:00<?, ?B/s]

[NeMo I 2026-07-22 07:02:09 mixins:184] Tokenizer SentencePieceTokenizer initialized with 800 tokens


[NeMo W 2026-07-22 07:02:10 modelPT:188] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath:
    - /home/sujith/asrTraining/finetuningDataset/manifest_train_hindi_cleaned.json
    sample_rate: 16000
    use_start_end_token: false
    batch_size: 16
    shuffle: true
    num_workers: 8
    pin_memory: true
    max_duration: 40
    min_duration: 0.1
    is_tarred: false
    tarred_audio_filepaths: null
    shuffle_n: 2048
    bucketing_strategy: synced_randomized
    bucketing_batch_size: null
    
[NeMo W 2026-07-22 07:02:10 modelPT:195] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setup the validation data loader(s). 
    Validation config : 
    manifest_filepath:
    - /home/sujith/asrTraining/finetun

[NeMo I 2026-07-22 07:02:14 rnnt_models:226] Using RNNT Loss : tdt
    Loss tdt_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0, 'durations': [0, 1, 2, 3, 4], 'sigma': 0.02, 'omega': 0.1}
[NeMo I 2026-07-22 07:02:14 rnnt_models:226] Using RNNT Loss : tdt
    Loss tdt_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0, 'durations': [0, 1, 2, 3, 4], 'sigma': 0.02, 'omega': 0.1}
[NeMo I 2026-07-22 07:02:14 rnnt_models:226] Using RNNT Loss : tdt
    Loss tdt_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0, 'durations': [0, 1, 2, 3, 4], 'sigma': 0.02, 'omega': 0.1}
[NeMo I 2026-07-22 07:02:15 save_restore_connector:285] Model EncDecRNNTBPEModel was successfully restored from /root/.cache/huggingface/hub/models--ARTPARK-IISc--Vaani-FastConformer-Hindi/snapshots/9fa335764565e4407838066d904fa101282744f4/Vaani-FastConformer-Hindi.nemo.
Model loaded on cuda as float32


## Config

Point `KATHBATH_ROOT` at wherever your attached Kaggle Dataset lands - check the file
listing printed in the first cell, or the "Data" panel on the right, to confirm the exact
path (it's usually `/kaggle/input/<your-dataset-slug>`, which may or may not itself
contain a `kathbath/` subfolder depending on how you zipped it).

`KATHBATH_ROOT/hindi/` is expected to contain:
- `manifest.json` - one JSON object per line: `{"audio_filepath": ..., "duration": ..., "text": ...}`
- an audio subfolder (commonly `wav/` or `wavs/`) - the exact name doesn't matter, the
  notebook indexes every `.wav`/`.flac`/`.mp3` file under the language folder by filename.

Sample count reported (Vistaar-filtered Kathbath test set):

| Language | Folder  | HF/NeMo model                          | Test samples |
|---|---|---|---|
| Hindi    | `hindi` | `ARTPARK-IISc/Vaani-FastConformer-Hindi` | 1,929 |

`Vaani-FastConformer` ships one checkpoint per language, so `LANGUAGES` below only ever
holds a single entry - swap the model repo id (in the cell above) and folder name (below)
if ARTPARK release a checkpoint for another language.


In [29]:
# ── KATHBATH_ROOT is already set by the download/extract cell above.
# If you're attaching a manually-uploaded Kaggle Dataset instead, uncomment and edit:
# KATHBATH_ROOT = "/kaggle/input/kathbath"

# (kathbath_folder_name, display_name) - only languages actually extracted above will work.
LANGUAGES = [
    ("hindi", "Hindi"),
]

# Set to an int (e.g. 300) to cap samples for a quick test run, or None to evaluate
# every sample in the manifest.json.
SAMPLES_PER_LANG = None

CHECKPOINT_EVERY = 100
CHECKPOINT_DIR = "/kaggle/working"


In [30]:
import json
import os
from pathlib import Path
from tqdm import tqdm
from jiwer import wer, cer


def read_manifest(manifest_path):
    """Reads a NeMo-style manifest: one JSON object per line. Falls back to a single
    JSON array if the file isn't line-delimited."""
    with open(manifest_path, "r", encoding="utf-8") as f:
        content = f.read().strip()
    entries = []
    try:
        for line in content.splitlines():
            line = line.strip()
            if not line:
                continue
            entries.append(json.loads(line))
    except json.JSONDecodeError:
        entries = json.loads(content)
    return entries


def build_audio_index(lang_dir: Path) -> dict:
    """Scans lang_dir recursively once and maps basename -> full path. Robust to any
    audio subfolder naming ('wav', 'wavs', etc.) and any nesting depth."""
    index = {}
    for ext in ("*.wav", "*.flac", "*.mp3"):
        for p in lang_dir.rglob(ext):
            index[p.name] = str(p)
    return index


def resolve_audio_path(lang_dir: Path, audio_filepath: str, audio_index: dict) -> str:
    """audio_filepath in the manifest may be absolute (from the original download
    machine), relative to some root above lang_dir (e.g. 'kathbath/hindi/wavs/x.wav'),
    or just a bare filename. Try direct candidates first, then fall back to the
    prebuilt basename index (handles any subfolder naming mismatch, e.g. 'wav' vs 'wavs')."""
    p = Path(audio_filepath)
    candidates = [
        p if p.is_absolute() else None,
        lang_dir / audio_filepath,
        lang_dir.parent / audio_filepath,  # manifest path already includes '<lang>/wavs/...'
        lang_dir / "wav" / p.name,
        lang_dir / "wavs" / p.name,
        lang_dir / p.name,
    ]
    for c in candidates:
        if c is not None and c.exists():
            return str(c)
    if p.name in audio_index:
        return audio_index[p.name]
    raise FileNotFoundError(
        f"Could not resolve audio file '{audio_filepath}' under {lang_dir} "
        f"(tried direct path candidates and a full basename index of {len(audio_index)} "
        f"audio files under this folder)."
    )


def checkpoint_path(lang_folder):
    return os.path.join(CHECKPOINT_DIR, f"checkpoint_{lang_folder}.json")


def save_checkpoint(idx, refs, preds, lang_folder):
    path = checkpoint_path(lang_folder)
    data = {
        "last_index": idx,
        "references": refs,
        "predictions": preds,
    }
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    os.replace(tmp, path)
    print(f"  \u2714 Checkpoint saved at sample {idx} \u2192 {path}")


def load_checkpoint(lang_folder):
    path = checkpoint_path(lang_folder)
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        print(f"Resuming {lang_folder} from checkpoint: {data['last_index'] + 1} samples already done.")
        return data["last_index"] + 1, data["references"], data["predictions"]
    return 0, [], []


def transcribe_one(wav_path):
    """NeMo's ASRModel.transcribe takes a list of audio file paths directly - Kathbath's
    wavs are already 16kHz mono, so no manual resampling/loading is needed here (unlike
    the IndicWav2Vec HF-pipeline notebook, which needs raw arrays)."""
    hyps = model.transcribe([wav_path], batch_size=1, verbose=False)
    hyp = hyps[0]
    return hyp.text if hasattr(hyp, "text") else str(hyp)


def evaluate_language(lang_folder, display_name, kathbath_root, n_samples=None):
    """Reads manifest.json for one language, runs Vaani-FastConformer inference over its
    audio, checkpointing periodically. Returns (references, predictions)."""
    print(f"\n{'='*70}\nEvaluating: {display_name}  (folder={lang_folder})\n{'='*70}")

    lang_dir = Path(kathbath_root) / lang_folder
    manifest_path = lang_dir / "manifest.json"
    if not manifest_path.exists():
        raise FileNotFoundError(f"manifest.json not found at {manifest_path} - check KATHBATH_ROOT.")

    entries = read_manifest(manifest_path)
    if n_samples is not None:
        entries = entries[:n_samples]
    total = len(entries)
    print(f"Found {total} entries in manifest.")

    print("Indexing audio files...")
    audio_index = build_audio_index(lang_dir)
    print(f"Indexed {len(audio_index)} audio files under {lang_dir}.")

    start_idx, references, predictions = load_checkpoint(lang_folder)

    last_idx = start_idx - 1
    for sample_idx in tqdm(range(start_idx, total), initial=start_idx, total=total, desc=display_name):
        entry = entries[sample_idx]
        reference_text = entry["text"]

        try:
            wav_path = resolve_audio_path(lang_dir, entry["audio_filepath"], audio_index)
            transcription = transcribe_one(wav_path)

            references.append(reference_text)
            predictions.append(transcription.strip())
            last_idx = sample_idx

        except (RuntimeError, FileNotFoundError) as e:
            print(f"Skipping sample {sample_idx} due to error: {e}.")
            continue

        if (sample_idx + 1) % CHECKPOINT_EVERY == 0:
            save_checkpoint(sample_idx, references, predictions, lang_folder)

    save_checkpoint(last_idx, references, predictions, lang_folder)
    print(f"Inference for {display_name} complete: {len(references)} samples evaluated.")
    return references, predictions


In [31]:
# ── Run evaluation for Hindi ────────────────────────────────────────────────
results = {}

for lang_folder, display_name in LANGUAGES:
    refs, preds = evaluate_language(
        lang_folder, display_name, KATHBATH_ROOT, n_samples=SAMPLES_PER_LANG
    )
    results[display_name] = {
        "references": refs,
        "predictions": preds,
    }



Evaluating: Hindi  (folder=hindi)
Found 1929 entries in manifest.
Indexing audio files...
Indexed 1929 audio files under /kaggle/working/kathbath_data/kathbath/hindi.
Resuming hindi from checkpoint: 1929 samples already done.


Hindi: 100%|██████████| 1929/1929 [00:00<?, ?it/s]

  ✔ Checkpoint saved at sample 1928 → /kaggle/working/checkpoint_hindi.json
Inference for Hindi complete: 1929 samples evaluated.


In [32]:
# ── Compute WER/CER per language and build a summary table ──────────────────
rows = []
for display_name, r in results.items():
    refs, preds = r["references"], r["predictions"]
    if not refs:
        print(f"No samples evaluated for {display_name}, skipping metrics.")
        continue
    lang_wer, lang_cer = wer(refs, preds), cer(refs, preds)
    rows.append({
        "Language": display_name,
        "N": len(refs),
        "WER %": round(lang_wer * 100, 2),
        "CER %": round(lang_cer * 100, 2),
    })

summary_df = pd.DataFrame(rows).set_index("Language")
print(summary_df.to_string())

out_csv = "/kaggle/working/vaani_fastconformer_kathbath_summary.csv"
summary_df.to_csv(out_csv)
print(f"\nSaved summary to {out_csv}")


             N  WER %  CER %
Language                    
Hindi     1929  12.49    4.3

Saved summary to /kaggle/working/vaani_fastconformer_kathbath_summary.csv


## Notes

- **Path issues:** if you get a `manifest.json not found` error, print
  `os.listdir(KATHBATH_ROOT)` to see the exact top-level folder name Kaggle mounted your
  dataset under, and adjust `KATHBATH_ROOT` accordingly (Kaggle sometimes nests an extra
  folder level depending on how the dataset was zipped/uploaded).
- **`audio_filepath` resolution:** manifests can have absolute paths from the original
  download machine, paths already prefixed with `kathbath/hindi/wavs/...`, or bare
  filenames - all of which break naive joining against `KATHBATH_ROOT`. The notebook
  scans the language folder once (`build_audio_index`) and matches every manifest entry
  by filename, so it's robust regardless of subfolder naming (`wav` vs `wavs`) or path
  prefixes in the manifest - no manifest editing needed.
- **Resuming across sessions:** `checkpoint_hindi.json` in `/kaggle/working` holds every
  reference/prediction gathered so far. If a session times out, save `/kaggle/working` as
  a Kaggle Dataset, start a new session, copy the checkpoint file back into
  `/kaggle/working`, and re-run - it resumes from `last_index + 1` automatically.
- **No temp-file round-trip needed:** the original Vaani/IndicVoices notebook wrote each
  HF-dataset audio array out to a temp `.wav` before calling `model.transcribe(...)`,
  because IndicVoices only gives you decoded arrays. Kathbath already ships real `.wav`
  files on disk, so `transcribe_one` just passes the resolved path straight to NeMo -
  one less moving part, and no cleanup needed.
- **bf16 → float32 fix:** some Vaani-FastConformer checkpoints store weights in `bf16`,
  which T4/P100 GPUs don't handle reliably across all ops. The model is loaded on CPU
  first and cast with `.float()` before moving to GPU, exactly as in the original
  IndicVoices notebook.
- **Single decode pass:** like the IndicWav2Vec notebook, this is one CTC decode per
  utterance - no CTC/RNNT split (that's specific to IndicConformer's dual-decoding
  multilingual checkpoint).
